# Bonus

🎯 84 feature’dan oluşan tam `ML_Houses_dataset.csv` dataset’iyle [buradan ulaşarak](https://d32aokrjazspmn.cloudfront.net/materials/ML_Houses_dataset.csv) serbestçe çalışabilirsiniz!

- Feature’ları inceleyin
- Uygun şekilde preprocess edin ve encode edin
- Feature engineering için beyin fırtınası yapın
- Bunları modelinize ekleyin
- Feature selection uygulayın

👇 Dosyayı yerel olarak `data` klasörüne kaydedin ve buradan içe aktarın.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://d32aokrjazspmn.cloudfront.net/materials/ML_Houses_dataset.csv"
df = pd.read_csv(url)

df.drop(columns=['Pesos', 'Id'], inplace=True, errors='ignore')

In [1]:
categorical_cols = df.select_dtypes(include=['object']).columns
df[categorical_cols] = df[categorical_cols].fillna('None')

numerical_cols = df.select_dtypes(exclude=['object']).columns
df[numerical_cols] = df[numerical_cols].fillna(df[numerical_cols].median())

NameError: name 'df' is not defined

In [ ]:
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']
df['TotalSF'] = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF']
df['TotalBaths'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])

In [ ]:
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

X_encoded = pd.get_dummies(X, drop_first=True, dtype=float)

scaler = RobustScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_encoded), columns=X_encoded.columns)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

NameError: name 'df' is not defined

ℹ️ Dataset’in açıklamasına mutlaka [buradan](https://drive.google.com/file/d/1qLxeQXufW_-KHOckpUweLPhitzjnP7H3/view?usp=sharing) referans verin.

In [ ]:
model = Ridge(alpha=10.0)
model.fit(X_train, y_train)
base_score = model.score(X_test, y_test)
print(f"Base Model R2 Score: {base_score:.4f}")

perm_importance = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=42)

importance_df = pd.DataFrame({
    'Feature' : X_test.columns,
    'Importance' : perm_importance.importances_mean
}).sort_values(by='Importance', ascending=False)

weak_features = importance_df[importance_df['Importance'] <= 0.001]['Feature'].tolist()
print(f"Toplam Feature: {len(X_test.columns)} | Çıkarılacak Zayıf Feature: {len(weak_features)}")

X_train_strong = X_train.drop(columns=weak_features)
X_test_strong = X_test.drop(columns=weak_features)

strong_model = Ridge(alpha=10.0)
strong_model.fit(X_train_strong, y_train)
strong_score = strong_model.score(X_test_strong, y_test)
print(f"Strong Model R2 Score (Seçilmiş Özelliklerle): {strong_score:.4f}")